# 처음 시작하는 CIFAR-10 백본 비교 실습

이 노트북은 이미지 분류를 처음 접하는 사람도 **모델 선택 → 데이터 확인 → 학습/검증 → 최종 테스트** 과정을 순서대로 실행해 볼 수 있도록 구성했습니다. 단일 뉴런부터 차근차근 시작하려면 먼저 `foundations.ipynb`를 실행하세요.

### 처음 실행하는 방법

1. 위 메뉴에서 **모든 셀 실행(Run All)** 을 선택합니다.
2. 처음에는 아래의 `MODEL_ID = "resnet18"`, `QUICK_RUN = True`를 그대로 사용합니다.
3. 그래프와 최종 정확도가 출력되면 전체 과정이 정상 동작한 것입니다.
4. 익숙해진 뒤 `MODEL_ID` 또는 `QUICK_RUN`만 변경해 다른 실험을 진행합니다.

> `QUICK_RUN=True`는 일부 데이터와 1 epoch만 사용하므로 정확도보다는 전체 실행 흐름을 확인하는 용도입니다.

## 0. 필요한 기능 불러오기

백본 구현을 하나씩 직접 import하지 않습니다. Registry가 `MODEL_ID`에 맞는 모델 모듈을 필요한 시점에 불러옵니다.

In [ ]:
# 모델 구조와 파라미터 수를 보기 위한 도구입니다.
from torchinfo import summary

# cifar10_lab의 공통 설정, 모델 Registry, 학습/평가 기능을 불러옵니다.
from cifar10_lab import (
    DataConfig,
    ExperimentConfig,
    TrainConfig,
    create_model,
    detect_environment,
    evaluate_model_detailed,
    experiment_id,
    format_model_catalog,
    get_lab_paths,
    load_cifar10_data,
    load_model_weights,
    resolve_data_dir,
    set_global_seed,
    train_model,
)

# 그래프를 그리는 기능은 시각화 모듈에서 따로 불러옵니다.
from cifar10_lab.visualization import (
    plot_test_results,
    plot_training_history,
    visualize_data_overview,
)

print("라이브러리 불러오기 완료")

## 1. 실험 설정하기 — 처음에는 두 값만 확인하세요

- `MODEL_ID`: 실행할 백본 이름입니다. 처음에는 `resnet18`을 권장합니다.
- `QUICK_RUN`: `True`이면 빠른 체험, `False`이면 전체 데이터를 사용합니다.

구조의 발전을 따라가려면 `perceptron`, `mlp`, `alexnet`, `resnet18` 순서로 실행해 보세요. 처음 빠른 백본 실험에는 `resnet18`, `mobilenet_v2`, `vgg11_bn`, `vit_tiny`가 적합합니다. 큰 모델은 메모리와 학습 시간이 많이 필요할 수 있습니다.

In [ ]:
# ============================================================
# 초보자는 아래 두 줄만 변경하면 됩니다.
# ============================================================
MODEL_ID = "resnet18"
QUICK_RUN = True

# 운영체제에 맞는 데이터·체크포인트·결과 저장 위치를 준비합니다.
paths = get_lab_paths(create=True)
data_root = str(resolve_data_dir())

if QUICK_RUN:
    # 빠른 체험 모드: 전체 흐름을 짧은 시간 안에 확인합니다.
    data_config = DataConfig(
        batch_size=64,
        val_ratio=0.1,
        seed=42,
        data_root=data_root,
        max_train_samples=2048,
        max_val_samples=512,
        max_test_samples=512,
    )
    train_config = TrainConfig(epochs=1, learning_rate=0.001)
    weight_dir = str(paths.checkpoints_dir / "quick")
else:
    # 전체 실험 모드: Train 45,000장, Validation 5,000장, Test 10,000장을 사용합니다.
    data_config = DataConfig(
        batch_size=64, val_ratio=0.1, seed=42, data_root=data_root
    )
    train_config = TrainConfig(epochs=10, learning_rate=0.001)
    weight_dir = str(paths.checkpoints_dir / "full")

# 아래 객체 하나가 모델, 데이터, 학습, 장치와 저장 위치를 모두 관리합니다.
config = ExperimentConfig(
    model_id=MODEL_ID,
    num_classes=10,          # CIFAR-10은 클래스가 10개입니다.
    image_size=32,           # CIFAR-10 이미지 크기는 32x32입니다.
    device="auto",        # CUDA → Apple MPS → CPU 순으로 자동 선택합니다.
    weight_dir=weight_dir,
    data=data_config,
    train=train_config,
)

# 데이터 분할뿐 아니라 모델 초기화와 학습 난수도 동일하게 고정합니다.
set_global_seed(config.data.seed, deterministic=True)
RUN_ID = experiment_id(config)

mode_name = "빠른 체험" if QUICK_RUN else "전체 학습"
print(f"실행 모드: {mode_name}")
print(f"선택 모델: {config.model_id}")
print(f"학습 epoch: {config.train.epochs}")
print(f"데이터 위치: {config.data.data_root}")
print(f"체크포인트 위치: {config.weight_dir}")
print(f"실험 ID: {RUN_ID}")

### 선택 가능한 모델 확인하기

아래 셀은 현재 32×32 CIFAR-10 파이프라인에서 바로 사용할 수 있는 모델 ID를 보여줍니다. `INSTALL GROUP`이 `transformers`인 모델은 노트북 설치 구성이 필요합니다.

In [ ]:
print(format_model_catalog(cifar10_ready_only=True))

## 2. 모델 생성과 구조 확인

Registry가 모델을 생성합니다. 출력의 마지막 shape이 `[1, 10]`이면 이미지 1장에 대해 10개 클래스 점수를 정상적으로 출력한다는 뜻입니다.

In [ ]:
# MODEL_ID에 해당하는 모델을 생성합니다.
model = create_model(
    config.model_id,
    num_classes=config.num_classes,
    image_size=config.image_size,
)

# 한 장의 가상 입력을 넣어 계층별 출력 크기와 파라미터 수를 확인합니다.
summary(
    model,
    input_size=(1, 3, config.image_size, config.image_size),
)

## 3. 실행 장치 확인

GPU가 있으면 CUDA, Apple Silicon에서는 MPS, 둘 다 없으면 CPU를 사용합니다. Windows 노트북에서는 데이터 로딩이 멈추는 일을 줄이기 위해 worker 기본값을 0으로 사용합니다.

In [ ]:
# 현재 컴퓨터에 맞는 장치와 DataLoader 옵션을 자동으로 결정합니다.
runtime = detect_environment(config.device, config.data.num_workers)
device = runtime.device

# 모델 파라미터를 선택한 장치로 이동합니다.
model = model.to(device)
print(f"실행 환경: {runtime}")

## 4. 데이터 분할과 이미지 확인

Train은 모델 학습, Validation은 최고 모델 선택, Test는 마지막 평가에만 사용합니다. 그래프에서 각 분할의 이미지 수와 augmentation이 적용된 Train 샘플을 확인하세요.

In [ ]:
# 설정값에 따라 CIFAR-10을 Train/Validation/Test로 준비합니다.
# 데이터가 없다면 첫 실행에서 자동으로 다운로드합니다.
trainloader, valloader, testloader, classes = load_cifar10_data(
    batch_size=config.data.batch_size,
    val_ratio=config.data.val_ratio,
    seed=config.data.seed,
    num_workers=runtime.num_workers,
    pin_memory=runtime.pin_memory,
    image_size=config.image_size,
    data_root=config.data.data_root,
    max_train_samples=config.data.max_train_samples,
    max_val_samples=config.data.max_val_samples,
    max_test_samples=config.data.max_test_samples,
)

# 분할 크기와 실제 학습 이미지를 그림으로 확인합니다.
visualize_data_overview(trainloader, valloader, testloader, classes)

## 5. 학습하고 Validation으로 최적 모델 선택

매 epoch마다 Train과 Validation의 loss·accuracy를 기록합니다. Validation accuracy가 가장 높은 모델만 저장합니다.

- Train과 Validation이 함께 좋아지면 정상적으로 학습 중입니다.
- Train만 좋아지고 Validation이 나빠지면 overfitting을 의심할 수 있습니다.
- 모델·seed·학습률·데이터 설정이 같으면 저장된 체크포인트를 불러와 학습을 반복하지 않습니다. 이어서 학습하려면 CLI의 `--resume`, 처음부터 다시 학습하려면 `--retrain`을 사용하세요.

In [ ]:
# 모델·seed·학습률·데이터 설정이 같은 실험 체크포인트만 불러옵니다.
checkpoint = load_model_weights(
    model,
    config.model_id,
    device=device,
    weight_dir=config.weight_dir,
    experiment_id=RUN_ID,
    expected_config=config,
)

if checkpoint is None:
    # 저장된 모델이 없을 때만 처음부터 학습합니다.
    history = train_model(
        model,
        trainloader,
        valloader,
        device,
        epochs=config.train.epochs,
        learning_rate=config.train.learning_rate,
        model_id=config.model_id,
        weight_dir=config.weight_dir,
        experiment_config=config,
        experiment_id=RUN_ID,
    )
else:
    # 저장된 학습 이력도 함께 복원합니다.
    history = checkpoint.get("history")

# Train/Validation의 loss와 accuracy 변화를 그래프로 확인합니다.
plot_training_history(history)

## 6. Test 데이터로 마지막 평가

Test는 모델을 선택하는 데 사용하지 않고 마지막에 한 번 평가합니다. Confusion matrix는 어떤 클래스를 서로 헷갈렸는지 보여주고, 오른쪽 막대그래프는 클래스별 정확도를 보여줍니다.

In [ ]:
# 최적 Validation checkpoint 상태의 모델을 Test 데이터로 평가합니다.
test_result = evaluate_model_detailed(
    model,
    testloader,
    device,
    num_classes=config.num_classes,
)

print(f"최종 Test accuracy: {test_result['accuracy']:.2f}%")
plot_test_results(test_result, classes)

## 다음 실험 아이디어

전체 과정이 정상 실행되었다면 한 번에 하나만 바꾸어 결과를 비교해 보세요.

1. `MODEL_ID`를 `perceptron`, `mlp`, `alexnet`으로 바꾸어 선형 모델→비선형 완전연결망→CNN의 차이를 비교합니다.
2. `MODEL_ID`를 `mobilenet_v2`로 바꾸어 ResNet18과 파라미터 수를 비교합니다.
3. `MODEL_ID`를 `vit_tiny`로 바꾸어 CNN과 Transformer의 학습 곡선을 비교합니다.
3. `QUICK_RUN=False`로 바꾸어 전체 데이터에서 학습합니다.
4. learning rate를 `0.001`에서 `0.0001`로 바꾸어 수렴 속도를 비교합니다.

> 공정한 비교를 위해 한 실험에서는 모델 외의 batch size, epoch, seed를 동일하게 유지하세요.